In [1]:
import pandas as pd
import re

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [6]:
download_service = Service()
driver = webdriver.Chrome(service=download_service)

#driver.get("https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs=0")
# driver.get("https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16")
# https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEyBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMtD9aDKZ+9GYcVUKukEj6XnJhNql
#https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEzBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMlWnHdgz3AaHVlQ3gBlfXfTWethI

sklep_opon_base_url = "https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs="
oponeo_base_url = "https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16"

In [3]:

import time

def hide_sklep_opon_popups():
    # Obsługa przycisku akceptacji ciasteczek
    try:
        btn_cookie = driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
        btn_cookie.click()
        print("Przycisk akceptacji ciasteczek został kliknięty.")
    except NoSuchElementException:
        print("Przycisk akceptacji ciasteczek nie został znaleziony.")
    except Exception as exception:
        print("Nie udało się kliknąć przycisku akceptacji ciasteczek:", exception)
    
    # Obsługa okna powiadomień w shadow DOM
    try:
        driver.execute_script("""
            const shadowHost = document.querySelector("body > div.gr-visual-prompt");
            if (shadowHost) {
                const shadowRoot = shadowHost.shadowRoot;
                const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
                if (closeButton) {
                    closeButton.click();
                    console.log("Okienko powiadomień zostało zamknięte.");
                } else {
                    console.log("Nie znaleziono przycisku zamknięcia powiadomień.");
                }
            } else {
                console.log("Okno powiadomień nie zostało znalezione.");
            }
        """)
    except Exception as exception:
        print("Nie udało się zamknąć okienka powiadomień:", exception)

def close_oponeo_privacy_popup():
    try:
        # Wyszukanie elementu przycisku "Odrzuć wszystkie"
        reject_button = driver.find_element(By.CSS_SELECTOR, "#consentsBar > div.buttonsContainer.container > div > span.reject")
        reject_button.click()
        print("Okienko prywatności zostało zamknięte.")
    except NoSuchElementException:
        print("Okienko prywatności nie jest widoczne lub zostało już zamknięte.")
    except Exception as e:
        print("Wystąpił błąd podczas zamykania okienka prywatności:", e)

In [5]:


def load_sklep_opon_tyre_data():
    try:
        # Znalezienie wszystkich elementów opon w sekcji listing-products-element
        opony_elements = driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
        
        # Iteracja przez każdy element opony
        for opona_element in opony_elements:
            # Słownik do przechowywania danych jednej opony
            opona_data = {}
    
            # Pobranie danych do słownika
            opona_data['name'] = opona_element.get_attribute('data-ee-product-properties').split(";")[0].split(":")[1]
            opona_data['brand'] = opona_element.get_attribute('data-ee-product-properties').split(";")[4].split(":")[1]
            opona_data['model'] = opona_element.get_attribute('data-ee-product-properties').split(";")[6].split(":")[1]
            opona_data['size'] = opona_element.get_attribute('data-ee-product-properties').split(";")[5].split(":")[1]
            
            # Znalezienie indeksu nośności i indeksu prędkości
            try:
                load_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="li"]').text
                speed_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="si"]').text
                opona_data['load_index'] = load_index
                opona_data['speed_index'] = speed_index
            except NoSuchElementException:
                opona_data['load_index'] = None
                opona_data['speed_index'] = None
            
            # Pobieranie szczegółowych informacji o etykiecie
            etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
            opona_data['fuel_index'] = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
            opona_data['wet_grip_index'] = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
            opona_data['noise_index'] = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
            
            # Pobieranie poziomu hałasu, uwzględniając wewnętrzny <span> z dB
            try:
                noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
                for noise_level_element in noise_level_elements:
                    text = noise_level_element.text
                    match = re.search(r'\d+', text)
                    if match:
                        noise_level = int(match.group())
                    else:
                        noise_level = None
                    opona_data['noise_level'] = noise_level
            except (NoSuchElementException, IndexError):
                opona_data['noise_level'] = None
            
            class_mapping = {
                "Premium": "Premium",
                "Średnia": "Średnia",
                "Średniej": "Średnia",
                "Ekonomiczna": "Ekonomiczna",
                "Ekonomicznej": "Ekonomiczna"
            }
            try:
                tyre_class_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
                # Dopasuj i wyczyść tekst
                opona_class = tyre_class_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
                opona_data['class'] = class_mapping.get(opona_class, opona_class)
            except NoSuchElementException:
                opona_data['class'] = None
            
            # Pobieranie oceny użytkownika (tekstowa wartość obok gwiazdek)
            try:
                user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
                opona_data['user_rating'] = float(user_rating_element.text.replace(",", "."))
            except NoSuchElementException:
                opona_data['user_rating'] = None
            
            opona_data['price'] = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
            
            # Dodajemy dane opony do listy
            sklep_opon_tyres_data.append(opona_data)
    finally:
        pass

sklep_opon_tyres_data = []
offset = 0

while True:
    url = f"{sklep_opon_base_url}{offset}"
    driver.get(url)
    time.sleep(4)
    
    if offset == 0:
        hide_sklep_opon_popups()
    
    load_sklep_opon_tyre_data()
    
    offset += 20
    # Warunek zatrzymania przy pustej stronie
    if len(driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')) == 0:
        print("Brak nowych danych. Koniec paginacji.")
        break
# Zamknięcie WebDrivera
driver.quit()

# Wydrukowanie listy wszystkich danych o oponach
df = pd.DataFrame(sklep_opon_tyres_data)
display(df)

Przycisk akceptacji ciasteczek nie został znaleziony.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price
0,Wintrac 205/55 R16 91 H,Vredestein,Wintrac,205/55 R16,91,H,C,B,B,70,Premium,5.4,376.99
1,Winguard Snow'G WH2 205/55 R16 91 H,Nexen,Winguard Snow'G WH2,205/55 R16,91,H,D,C,B,70,Średnia,5.3,310.00
2,SP901 205/55 R16 91 H,Austone,SP901,205/55 R16,91,H,D,D,B,72,Ekonomiczna,5.1,224.99
3,Frigo HP2 205/55 R16 91 H,Dębica,Frigo HP2,205/55 R16,91,H,C,C,B,72,Ekonomiczna,5.2,283.00
4,Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71,Ekonomiczna,5.1,239.00
5,Winter i*cept RS3 W462 205/55 R16 91 T,Hankook,Winter i*cept RS3 W462,205/55 R16,91,T,C,B,B,72,Premium,5.3,322.00
6,Snowproof 1 205/55 R16 91 H,Nokian Tyres,Snowproof 1,205/55 R16,91,H,C,B,B,70,Premium,5.4,335.00
7,SW608 205/55 R16 91 H,Goodride,SW608,205/55 R16,91,H,C,C,B,72,Ekonomiczna,5.1,244.99
8,Snowproof 2 205/55 R16 91 H,Nokian Tyres,Snowproof 2,205/55 R16,91,H,C,B,A,69,Premium,5.6,385.00
9,Z507 205/55 R16 91 V,Goodride,Z507,205/55 R16,91,V,C,C,B,72,Ekonomiczna,5.1,227.00


In [31]:
def load_oponeo_tyre_data(driver):
    """
    Pobiera dane opon z bieżącej strony i zwraca je jako listę słowników.
    """
    tires_data = []
    products = driver.find_elements(By.CLASS_NAME, "product")
    
    for product in products:
        try:
            # Wyciąganie linku i nazwy produktu z `productName`, jeśli dostępne
            try:
                link_element = product.find_element(By.CSS_SELECTOR, ".productName a")
                nazwa = link_element.get_attribute("title")
            except NoSuchElementException:
                # Jeśli nie znaleziono elementu `a` wewnątrz `productName`
                nazwa = product.find_element(By.CLASS_NAME, "productName").text

            noise = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text
            match = re.search(r'\d+', noise)
            if match:
                noise_level = int(match.group())
            else:
                noise_level = int(product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[1].replace("dB", "").strip())
             
             # Wyciąganie pozostałych szczegółów z dodaną walidacją dla `wet_grip_index`
            noise_index = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[0] 
            noise_index_text = noise_index if noise_index in {"A", "B", "C", "D", "E", "F"} and len(noise_index) == 1 else None   
                
             # Pobieranie oceny użytkownika (lub None jeśli brak oceny)
            try:
                user_rating = product.find_element(By.CSS_SELECTOR, ".productRating .note").text
            except NoSuchElementException:
                user_rating = None
                
            # Wyciąganie pozostałych szczegółów
            tire_info = {
                "name": nazwa,
                "brand": product.find_element(By.CLASS_NAME, "producerName").text,
                "model": product.find_element(By.CLASS_NAME, "modelName").text,
                "size": product.find_element(By.CLASS_NAME, "modelSize").text,
                "load_index": product.find_element(By.XPATH, ".//span[@data-tp='TireLoadIndex']/em").text,
                "speed_index": product.find_element(By.XPATH, ".//span[@data-tp='TireSpeedIndex']/em").text,
                "fuel_index": product.find_element(By.CSS_SELECTOR, ".icon-fuel em").text,
                "wet_grip_index": product.find_element(By.CSS_SELECTOR, ".icon-rain em").text,
                "noise_index": noise_index_text,
                "noise_level": noise_level,
                "class": product.find_element(By.CLASS_NAME, "class").text.replace("KLASA ", "").capitalize(),
                "user_rating": user_rating,
                "price": product.find_element(By.CLASS_NAME, "priceValue").text,
            }
            
            # Dodajemy słownik do listy
            tires_data.append(tire_info)

        except NoSuchElementException:
            pass
            
    return tires_data

def go_to_next_page(web_driver, current_page_number):
    """
    Przechodzi do następnej strony na podstawie numeru strony. 
    Zwraca True, jeśli przejście się powiodło, w przeciwnym razie False.
    """
    try:
        # Wyszukiwanie przycisku z numerem strony
        print(f"finding page {current_page_number}")
        next_page_button = web_driver.find_element(By.ID, f"_ctPgrp_pi{current_page_number}i")
        print(f"page found {current_page_number}")
        next_page_button.click()
        # Opcjonalne odczekanie na załadowanie strony
        time.sleep(2)
        return True
    except NoSuchElementException:
        print(f"no page {current_page_number}")
        return False

# Kod główny do pobierania danych ze wszystkich stron
driver.get(oponeo_base_url)
close_oponeo_privacy_popup()

all_tires_data = []

page_number = 1
while True:
    # Pobranie danych z bieżącej strony
    tires_data = load_oponeo_tyre_data(driver)
    all_tires_data.extend(tires_data)
    
    # Przejście do następnej strony
    page_number += 1
    if not go_to_next_page(driver, page_number):
        break

# Wyświetlenie zebranych danych w DataFrame
df2 = pd.DataFrame(all_tires_data)
display(df2)

# Zamknięcie drivera
# driver.quit()

Okienko prywatności nie jest widoczne lub zostało już zamknięte.
finding page 2
page found 2
finding page 3
page found 3
finding page 4
page found 4
finding page 5
page found 5
finding page 6
page found 6
finding page 7
page found 7
finding page 8
page found 8
finding page 9
page found 9
finding page 10
no page 10


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price
0,Bridgestone Blizzak LM005 205/55 R16 91 H,Bridgestone,Blizzak LM005,205/55 R16,91,H,C,A,B,71,Premium,"4,7",459
1,Michelin Alpin 7 205/55 R16 91 H,Michelin,Alpin 7,205/55 R16,91,H,C,B,B,71,Premium,"4,8",467
2,Dębica Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71,Ekonomiczna,"4,2",252
3,Kormoran Snow 205/55 R16 91 H,Kormoran,Snow,205/55 R16,91,H,D,C,B,72,Ekonomiczna,"4,5",249
4,Firemax FM805+ 205/55 R16 91 H,Firemax,FM805+,205/55 R16,91,H,D,C,A,67,Ekonomiczna,"4,4",209
...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,Goodyear UG Performance 2 205/55 R16 91 H RUN ...,Goodyear,UG Performance 2,205/55 R16,91,H,D,C,B,72,Premium,"4,3",778
225,Bridgestone Blizzak LM005 205/55 R16 94 V XL,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71,Premium,"4,7",854
226,Barum Polaris 5 205/55 R16 94 V XL,Barum,Polaris 5,205/55 R16,94,V,C,C,B,72,Ekonomiczna,"4,4",854
227,Pirelli SottoZero Serie 3 205/55 R16 91 H RUN ...,Pirelli,SottoZero Serie 3,205/55 R16,91,H,D,B,B,72,Premium,"4,6",972


In [32]:
# Weryfikacja poprawnych wartości w każdej kolumnie
print("sklep opon")
for column in df.columns:
    print(column, df[column].unique())

print("oponeo")
for column in df2.columns:
    print(column, df2[column].unique())

sklep opon
name ['Wintrac 205/55 R16 91 H' "Winguard Snow'G WH2 205/55 R16 91 H"
 'SP901 205/55 R16 91 H' 'Frigo HP2 205/55 R16 91 H'
 'Frigo 2 205/55 R16 91 T' 'Winter i*cept RS3 W462 205/55 R16 91 T'
 'Snowproof 1 205/55 R16 91 H' 'SW608 205/55 R16 91 H'
 'Snowproof 2 205/55 R16 91 H' 'Z507 205/55 R16 91 V'
 'Wintercraft WP52 205/55 R16 91 H' 'WinterContact TS 870 205/55 R16 91 H'
 'Polaris 6 205/55 R16 91 T' 'Snowproof 2 205/55 R16 91 T'
 'Eurowinter HS02 205/55 R16 91 H' 'WinterExpert 205/55 R16 91 T'
 'WinterExpert 205/55 R16 91 H' 'Winter Sport 5 205/55 R16 91 H'
 'Blizzak LM005 205/55 R16 91 H' 'Winterhawk 4 205/55 R16 91 H'
 'Winter i*cept RS3 W462 205/55 R16 94 H'
 "Winguard Snow'G3 WH21 205/55 R16 91 T" 'Snowproof 2 205/55 R16 94 H']
brand ['Vredestein' 'Nexen' 'Austone' 'Dębica' 'Hankook' 'Nokian Tyres'
 'Goodride' 'Kumho' 'Continental' 'Barum' 'Falken' 'Uniroyal' 'Dunlop'
 'Bridgestone' 'Firestone']
model ['Wintrac' "Winguard Snow'G WH2" 'SP901' 'Frigo HP2' 'Frigo 2'
 'Wint